In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "granite4.2:3b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

# Verificar integridad
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Cliente local sin proxy
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos = json.load(r)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Granite 4.2 3B no está instalado."

# Consultar capacidades
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as r:
    detalles = json.load(r)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: granite4.2:3b-q4_K_M
Identificador: 40577dc168a3
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['tools', 'thinking', 'completion']
Thinking: None
Parámetros: temperature                    1
top_p                          0.95

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-G42-3B-01"

# Caso diagnóstico utilizado con los modelos anteriores
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Recuperar configuración V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO
solicitud["think"] = False

# Mismos parámetros utilizados con Granite 4.2 8B
solicitud["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1.0,
    "top_p": 0.95
})

solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# Comprobar que no existe una ejecución anterior
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r["case_id"] == CASO_ID for r in anteriores
), "El diagnóstico ya está registrado."

# Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "granite42_3b_directa_v1",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

# Ejecutar en Ollama local
inicio = time.perf_counter()

try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Guardar la ejecución
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

# Mostrar resultados
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Estado: ok
Tiempo cliente: 9.673 s
Tokens entrada: 438
Tokens generados: 608
Terminación: stop
Razonamiento separado: False
total_duration 9.671 s
load_duration 4.589 s
prompt_eval_duration 0.101 s
eval_duration 4.974 s

RESPUESTA FINAL:

Evidencia:  
[E1] Autor X tiene un artículo sobre bases de datos académicas.  
[E2] Autor Y tiene un artículo sobre recuperación de información.

Interpretación:  
La predicción recibida es favorable (simulada), lo que indica que se espera una coautoría entre Autor X y Autor Y. Las dos evidencias documentan la experiencia individual de cada autor en áreas relacionadas pero distintas: Autor X ha trabajado en bases de datos académicas, mientras que Autor Y ha desarrollado investigaciones en recuperación de información. Estas áreas comparten un enfoque común en el campo de la información y la búsqueda, pero no implican necesariamente una colaboración directa.

Limitaciones:  
No se proporciona ninguna evidencia de colaboración previa (por ejemplo, public

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "granite4.2:3b-q4_K_M"
ARCHIVO = "resultados.jsonl"
API_LOCAL = "http://127.0.0.1:11434"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["api_url"] == API_LOCAL + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G42-3B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is False

opciones = diagnostico["solicitud"]["options"]

assert opciones["num_ctx"] == 4096
assert opciones["num_predict"] == 2048
assert opciones["temperature"] == 1.0
assert opciones["top_p"] == 0.95

# 3. Comprobar modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# No repetir ejecuciones ya registradas
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "granite42_3b_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar los parámetros del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "granite42_3b_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=300) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar inmediatamente cada resultado
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 899 tokens | 9.404 s
S-006 | ok | 775 tokens | 8.465 s
S-007 | ok | 809 tokens | 7.377 s
S-008 | ok | 721 tokens | 6.563 s
S-009 | ok | 642 tokens | 5.828 s
S-010 | ok | 781 tokens | 7.435 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "granite4.2:3b-q4_K_M"
API = "http://127.0.0.1:11434"

# 1. Recuperar el diagnóstico directo
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line) for line in f if line.strip()
    ]

assert not any(
    r["case_id"] == "DIAG-G42-3B-02"
    for r in registros
), "El diagnóstico ya existe."

anterior = next(
    r for r in registros
    if r["case_id"] == "DIAG-G42-3B-01"
)

assert anterior["estado"] == "ok"
assert anterior["solicitud"]["model"] == MODELO
assert anterior["solicitud"]["think"] is False

# 2. Copiar la solicitud y activar razonamiento
solicitud = copy.deepcopy(anterior["solicitud"])

solicitud["think"] = True

solicitud["options"].update({
    "num_ctx": 8192,
    "num_predict": 4096,
    "temperature": 1.0,
    "top_p": 0.95
})

# 3. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_actual = next(
    m for m in instalados if m["name"] == MODELO
)

assert modelo_actual["digest"] == anterior["modelo_digest"]

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-G42-3B-02",
    "grupo": "diagnostico",
    "configuracion_modelo": "granite42_3b_thinking_v1",
    "referencia": "DIAG-G42-3B-01",
    "modelo_digest": modelo_actual["digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico local
try:
    peticion = Request(
        API + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
        and mensaje.get("thinking")
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar sin sobrescribir resultados
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

# 7. Mostrar resultados
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))

else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: ok
Tiempo cliente: 22.511 s
Tokens entrada: 438
Tokens generados: 2109
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

Evidencia
[E1] Autor X tiene un artículo sobre bases de datos académicas.
[E2] Autor Y tiene un artículo sobre recuperación de información.

Interpretación
La predicción recibida es favorable (simulada). Las dos declaraciones documentan que cada autor ha publicado un artículo en áreas temáticamente relacionadas (bases de datos y recuperación de información). Esta semejanza temática sugiere una posibilidad de colaboración, pero la evidencia no muestra colaboración previa, proyectos conjuntos ni intercambios directos entre los autores. En consecuencia, la predicción favorable está parcialmente respaldada por la evidencia proporcionada.

Limitaciones
- La evidencia consiste únicamente en la descripción de publicaciones individuales; no contiene información sobre contactos, colaboración previa, proyectos compartidos o motivaciones que podrían lleva

In [5]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "granite4.2:3b-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["api_url"] == API + "/api/chat"

# 2. Recuperar resultados anteriores
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line) for line in f if line.strip()
    ]

directos = {
    r["case_id"]: r
    for r in anteriores
    if r.get("configuracion_modelo") == "granite42_3b_directa_v1"
    and r.get("grupo") == "comparacion"
}

assert len(directos) == 6

# Diagnóstico exitoso con razonamiento
diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-G42-3B-02"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["think"] is True
assert diagnostico["respuesta_original"]["done_reason"] == "stop"

opciones_thinking = copy.deepcopy(
    diagnostico["solicitud"]["options"]
)

assert opciones_thinking["num_ctx"] == 8192
assert opciones_thinking["num_predict"] == 4096
assert opciones_thinking["temperature"] == 1.0
assert opciones_thinking["top_p"] == 0.95

# 3. Verificar el modelo instalado
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "granite42_3b_thinking_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Copiar exactamente la solicitud original del caso
        solicitud = copy.deepcopy(
            directos[case_id]["solicitud"]
        )

        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Activar razonamiento con las opciones del diagnóstico
        solicitud["think"] = True
        solicitud["options"] = copy.deepcopy(opciones_thinking)

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "granite42_3b_thinking_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
                and mensaje.get("thinking")
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar inmediatamente cada ejecución
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 1705 tokens | 19.215 s
S-006 | ok | 1279 tokens | 13.958 s
S-007 | ok | 2257 tokens | 22.383 s
S-008 | ok | 1599 tokens | 17.255 s
S-009 | ok | 1387 tokens | 14.833 s
S-010 | ok | 1173 tokens | 12.55 s

Archivo actualizado: resultados.jsonl
